# Text Classification

## Learning Objectives
1. Build a TF-IDF baseline from scratch using NumPy and logistic regression with gradient descent
2. Implement an LSTM text classifier in PyTorch for multi-class classification
3. Handle class imbalance using class weights and oversampling, comparing macro-F1 outcomes
4. Tune decision thresholds, build multi-label classifiers, and benchmark TF-IDF vs CNN vs LSTM

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: TF-IDF + Logistic Regression Baseline in NumPy

TF-IDF (Term Frequency–Inverse Document Frequency) is a sparse, interpretable feature
representation that down-weights common words and up-weights discriminative ones.

Here we build TF-IDF from scratch — no sklearn — then train a logistic regression
classifier with mini-batch gradient descent on a synthetic binary sentiment corpus.

In [ ]:
# Level 1: TF-IDF from scratch + logistic regression with gradient descent

# ── Synthetic corpus ────────────────────────────────────────────────────────
POS_WORDS = ['good', 'great', 'excellent', 'happy', 'love', 'best', 'amazing', 'nice']
NEG_WORDS = ['bad',  'terrible', 'awful',    'hate',  'poor', 'worst', 'boring',  'sad']
NEUTRAL_WORDS = ['the', 'a', 'is', 'this', 'was', 'it', 'very', 'really', 'quite']

rng = np.random.default_rng(0)

def make_sentence(label: int, n_words: int = 8) -> list:
    """Generate a word list biased toward positive or negative sentiment."""
    signal_pool = POS_WORDS if label == 1 else NEG_WORDS
    words = []
    for _ in range(n_words):
        r = rng.random()
        if r < 0.50:
            words.append(rng.choice(signal_pool))
        else:
            words.append(rng.choice(NEUTRAL_WORDS))
    return words

raw_corpus = []
y_raw = []
for _ in range(50):
    raw_corpus.append(make_sentence(1))
    y_raw.append(1)
for _ in range(50):
    raw_corpus.append(make_sentence(0))
    y_raw.append(0)

y_raw = np.array(y_raw)

# ── Build vocabulary ────────────────────────────────────────────────────────
from collections import Counter

def build_vocab(corpus: list) -> dict:
    """Map unique words to integer indices."""
    counts = Counter(w for sent in corpus for w in sent)
    return {w: i for i, (w, _) in enumerate(counts.most_common())}

vocab = build_vocab(raw_corpus)
V = len(vocab)
N = len(raw_corpus)
print(f"Corpus: {N} sentences  |  Vocabulary: {V} words")

# ── Compute TF-IDF ──────────────────────────────────────────────────────────
def compute_tfidf(corpus: list, vocab: dict) -> np.ndarray:
    """
    Compute TF-IDF matrix [N, V].
      TF(t,d)  = count(t in d) / len(d)
      IDF(t)   = log(N / df(t))  where df(t) = number of docs containing t
    """
    V_size = len(vocab)
    tf = np.zeros((len(corpus), V_size))
    for i, sent in enumerate(corpus):
        for w in sent:
            if w in vocab:
                tf[i, vocab[w]] += 1
        tf[i] /= (len(sent) + 1e-8)

    df = (tf > 0).sum(axis=0)
    idf = np.log((len(corpus) + 1) / (df + 1))   # +1 smoothing
    return tf * idf

X_tfidf = compute_tfidf(raw_corpus, vocab)  # [100, V]
print(f"TF-IDF matrix shape: {X_tfidf.shape}")

# ── Logistic Regression with gradient descent ────────────────────────────────
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def binary_cross_entropy(y_hat: np.ndarray, y: np.ndarray) -> float:
    return -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))

# Train/test split
split = 80
X_tr_tf, X_te_tf = X_tfidf[:split], X_tfidf[split:]
y_tr, y_te = y_raw[:split], y_raw[split:]

# Normalise features
mu, std = X_tr_tf.mean(0), X_tr_tf.std(0) + 1e-8
X_tr_n = (X_tr_tf - mu) / std
X_te_n = (X_te_tf - mu) / std

w = np.zeros(V)
b = 0.0
lr_gd = 0.05
losses = []

for step in range(300):
    z    = X_tr_n @ w + b
    yhat = sigmoid(z)
    grad_w = X_tr_n.T @ (yhat - y_tr) / len(y_tr)
    grad_b = (yhat - y_tr).mean()
    w -= lr_gd * grad_w
    b -= lr_gd * grad_b
    if step % 50 == 0:
        losses.append(binary_cross_entropy(yhat, y_tr))

test_preds = (sigmoid(X_te_n @ w + b) >= 0.5).astype(int)
acc = accuracy_score(y_te, test_preds)
print(f"TF-IDF + Logistic Regression test accuracy: {acc:.3f}")
print(f"Loss at step 0: {losses[0]:.4f}  |  step 250: {losses[-1]:.4f}")

## Level 2: LSTM Text Classifier for 3-Class Topic Classification

Long Short-Term Memory networks process token sequences recurrently, capturing
sequential dependencies that bag-of-words methods miss.

Our synthetic 3-class corpus mimics topic categories:
- Class 0 — Technology: `code, data, model, train, network, ...`
- Class 1 — Sports: `team, game, score, player, win, ...`
- Class 2 — Food: `eat, cook, taste, recipe, flavor, ...`

We use a 70/15/15 train/val/test split and monitor validation loss.

In [ ]:
# Level 2: LSTM 3-class text classifier

# ── Vocabulary and synthetic dataset ────────────────────────────────────────
TOPIC_WORDS = {
    0: ['code', 'data', 'model', 'train', 'network', 'algo', 'layer', 'loss'],
    1: ['team', 'game', 'score', 'player', 'win',  'ball',  'match',  'goal'],
    2: ['eat',  'cook', 'taste', 'recipe', 'food', 'flavor','meal',   'chef'],
}
FILLER_WORDS = ['the', 'a', 'is', 'was', 'it', 'this', 'that', 'very']
PAD_IDX = 0

rng2 = np.random.default_rng(1)

def make_topic_sentence(cls: int, n_words: int = 10) -> list:
    """Generate a word sequence with topic-specific signal words."""
    words = []
    for _ in range(n_words):
        if rng2.random() < 0.55:
            words.append(rng2.choice(TOPIC_WORDS[cls]))
        else:
            words.append(rng2.choice(FILLER_WORDS))
    return words

def build_vocab_multi(sentences: list) -> dict:
    counts = Counter(w for s in sentences for w in s)
    return {w: i+1 for i, (w, _) in enumerate(counts.most_common())}  # 0=PAD

def encode_sentence(s: list, vocab: dict, max_len: int = 12) -> list:
    ids = [vocab.get(w, 0) for w in s]
    # Pad / truncate
    ids = ids[:max_len] + [PAD_IDX] * max(0, max_len - len(ids))
    return ids

# Build corpus: 50 sentences per class = 150 total
all_sents, all_labels = [], []
for cls in range(3):
    for _ in range(50):
        all_sents.append(make_topic_sentence(cls))
        all_labels.append(cls)

topic_vocab = build_vocab_multi(all_sents)
MAX_LEN = 12
VOCAB_SZ = len(topic_vocab) + 1  # +1 for PAD

# Encode and shuffle
X_enc = np.array([encode_sentence(s, topic_vocab) for s in all_sents])
y_enc = np.array(all_labels)
perm  = np.random.permutation(len(X_enc))
X_enc, y_enc = X_enc[perm], y_enc[perm]

# 70/15/15 split
n = len(X_enc)
t1, t2 = int(0.70 * n), int(0.85 * n)
X_tr_, X_va_, X_te_ = X_enc[:t1], X_enc[t1:t2], X_enc[t2:]
y_tr_, y_va_, y_te_ = y_enc[:t1], y_enc[t1:t2], y_enc[t2:]

def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(y, dtype=torch.long))

Xtr_t, ytr_t = to_tensor(X_tr_, y_tr_)
Xva_t, yva_t = to_tensor(X_va_, y_va_)
Xte_t, yte_t = to_tensor(X_te_, y_te_)
print(f"Train: {len(Xtr_t)}  Val: {len(Xva_t)}  Test: {len(Xte_t)}")

# ── LSTM classifier ──────────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    """Single-layer LSTM text classifier using last hidden state."""
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_classes: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc    = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed(x)
        _, (h, _) = self.lstm(emb)           # h: [1, B, H]
        return self.fc(h.squeeze(0))         # [B, num_classes]

lstm_model = LSTMClassifier(VOCAB_SZ, embed_dim=32, hidden_dim=64, num_classes=3).to(device)
opt_lstm = optim.Adam(lstm_model.parameters(), lr=1e-3)
ce_loss  = nn.CrossEntropyLoss()

train_losses, val_accs = [], []
for epoch in range(80):
    lstm_model.train()
    logits = lstm_model(Xtr_t.to(device))
    loss   = ce_loss(logits, ytr_t.to(device))
    opt_lstm.zero_grad(); loss.backward(); opt_lstm.step()
    train_losses.append(loss.item())

    lstm_model.eval()
    with torch.no_grad():
        val_logits = lstm_model(Xva_t.to(device))
        val_acc = (val_logits.argmax(1) == yva_t.to(device)).float().mean().item()
    val_accs.append(val_acc)

test_logits = lstm_model(Xte_t.to(device))
test_acc = accuracy_score(yte_t.numpy(), test_logits.argmax(1).cpu().numpy())
print(f"LSTM 3-class test accuracy: {test_acc:.3f}")
print(f"Final val accuracy: {val_accs[-1]:.3f}")

# ── CNN 1-D baseline (for Comparison cell) ──────────────────────────────────
class CNNClassifier(nn.Module):
    """1D convolutional text classifier (kernel size 3)."""
    def __init__(self, vocab_size: int, embed_dim: int,
                 n_filters: int, num_classes: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv  = nn.Conv1d(embed_dim, n_filters, kernel_size=3, padding=1)
        self.fc    = nn.Linear(n_filters, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed(x).permute(0, 2, 1)   # [B, D, T] for Conv1d
        feat = torch.relu(self.conv(emb))        # [B, F, T]
        pooled = feat.max(dim=2).values          # global max-pool [B, F]
        return self.fc(pooled)

cnn_model = CNNClassifier(VOCAB_SZ, embed_dim=32, n_filters=64, num_classes=3).to(device)
opt_cnn   = optim.Adam(cnn_model.parameters(), lr=1e-3)

for epoch in range(80):
    cnn_model.train()
    logits_c = cnn_model(Xtr_t.to(device))
    loss_c   = ce_loss(logits_c, ytr_t.to(device))
    opt_cnn.zero_grad(); loss_c.backward(); opt_cnn.step()

cnn_test_acc = accuracy_score(
    yte_t.numpy(),
    cnn_model(Xte_t.to(device)).argmax(1).detach().cpu().numpy())
print(f"CNN  3-class test accuracy: {cnn_test_acc:.3f}")

## Real-World Example 1: Handling Class Imbalance

Production datasets are rarely balanced. We create an imbalanced variant
(80% class 0, 10% class 1, 10% class 2) and compare three strategies:
1. **No weighting** — vanilla cross-entropy ignores imbalance
2. **Class weights** — scale the loss inversely proportional to class frequency
3. **Oversampling** — duplicate minority examples to rebalance the training set

Evaluation: **macro-F1** (treats all classes equally regardless of support).

In [ ]:
# Real-World Example 1: Class imbalance — weights vs oversampling

def make_imbalanced_dataset(n_majority: int = 120,
                             n_minority: int = 15) -> tuple:
    """
    Build an imbalanced 3-class corpus:
      class 0: n_majority examples, class 1 & 2: n_minority each
    """
    sents, labels = [], []
    for cls in range(3):
        n = n_majority if cls == 0 else n_minority
        for _ in range(n):
            sents.append(make_topic_sentence(cls))
            labels.append(cls)
    X = np.array([encode_sentence(s, topic_vocab) for s in sents])
    y = np.array(labels)
    perm = np.random.permutation(len(X))
    return X[perm], y[perm]

X_imb, y_imb = make_imbalanced_dataset()
n_imb = len(X_imb)
split80 = int(0.80 * n_imb)
Xi_tr, Xi_te = X_imb[:split80], X_imb[split80:]
yi_tr, yi_te = y_imb[:split80], y_imb[split80:]

Xi_tr_t = torch.tensor(Xi_tr, dtype=torch.long)
Xi_te_t = torch.tensor(Xi_te, dtype=torch.long)
yi_tr_t = torch.tensor(yi_tr, dtype=torch.long)
yi_te_t = torch.tensor(yi_te, dtype=torch.long)

def train_and_eval(class_weight_tensor=None,
                   oversample: bool = False,
                   n_epochs: int = 100,
                   label: str = '') -> float:
    """Train LSTMClassifier and return macro-F1 on test set."""
    if oversample:
        # Oversample minority classes to match majority count
        majority_n = (yi_tr == 0).sum()
        over_X, over_y = list(Xi_tr), list(yi_tr)
        for cls in [1, 2]:
            idx = np.where(yi_tr == cls)[0]
            need = majority_n - len(idx)
            sampled = np.random.choice(idx, need, replace=True)
            over_X.extend(Xi_tr[sampled].tolist())
            over_y.extend([cls] * need)
        X_t = torch.tensor(over_X, dtype=torch.long)
        y_t = torch.tensor(over_y, dtype=torch.long)
    else:
        X_t, y_t = Xi_tr_t, yi_tr_t

    model = LSTMClassifier(VOCAB_SZ, 32, 64, 3).to(device)
    weight = class_weight_tensor.to(device) if class_weight_tensor is not None else None
    criterion = nn.CrossEntropyLoss(weight=weight)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    for _ in range(n_epochs):
        model.train()
        logits = model(X_t.to(device))
        loss   = criterion(logits, y_t.to(device))
        opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(Xi_te_t.to(device)).argmax(1).cpu().numpy()
    f1 = f1_score(yi_te_t.numpy(), preds, average='macro', zero_division=0)
    print(f"  {label:35s}: macro-F1 = {f1:.3f}")
    return f1

# Class frequency weights: inversely proportional to count
counts = np.bincount(yi_tr, minlength=3).astype(float)
class_weights = torch.tensor((counts.sum() / (3 * counts)).astype(np.float32))

print("Training with three strategies on imbalanced data:")
f1_none    = train_and_eval(label='No weighting (baseline)')
f1_weights = train_and_eval(class_weight_tensor=class_weights, label='Class weights')
f1_over    = train_and_eval(oversample=True, label='Oversampling minority classes')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['No weighting', 'Class weights', 'Oversampling'],
       [f1_none, f1_weights, f1_over],
       color=['crimson', 'steelblue', 'darkorange'])
ax.set_ylabel('Macro-F1')
ax.set_title('Handling Class Imbalance (LSTM Classifier)')
ax.set_ylim(0, 1)
for i, v in enumerate([f1_none, f1_weights, f1_over]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/06_rw1_imbalance.png', dpi=100)
plt.close()
print("Plot saved to /tmp/06_rw1_imbalance.png")

## Real-World Example 2: Decision Threshold Tuning

Binary classifiers output a probability (after sigmoid). The default threshold of 0.5
is rarely optimal — you might want to maximise precision, recall, or F1 depending
on the cost of each error type.

We sweep threshold ∈ [0.1, 0.9], compute precision/recall/F1 at each point,
and identify the F1-maximising threshold.

In [ ]:
# Real-World Example 2: Decision threshold tuning — precision/recall curve

from sklearn.metrics import precision_score, recall_score

# Collapse to binary: class 0 vs. class 1+2
X_bin = np.concatenate([X_tr_[:70], X_tr_[70:105]])
y_bin_raw = np.concatenate([np.zeros(70, dtype=int), np.ones(35, dtype=int)])
X_bin_te  = X_te_[:30]
y_bin_te  = (y_te_[:30] > 0).astype(int)  # 0 vs rest

# Train a binary LSTM
bin_model = LSTMClassifier(VOCAB_SZ, 32, 64, 2).to(device)
opt_bin   = optim.Adam(bin_model.parameters(), lr=1e-3)
Xb_tr_t   = torch.tensor(X_bin, dtype=torch.long)
yb_tr_t   = torch.tensor(y_bin_raw, dtype=torch.long)

for _ in range(100):
    bin_model.train()
    logits_b = bin_model(Xb_tr_t.to(device))
    loss_b   = ce_loss(logits_b, yb_tr_t.to(device))
    opt_bin.zero_grad(); loss_b.backward(); opt_bin.step()

# Extract probabilities for class 1 on test set
bin_model.eval()
with torch.no_grad():
    logits_te = bin_model(torch.tensor(X_bin_te, dtype=torch.long).to(device))
    probs = torch.softmax(logits_te, dim=1)[:, 1].cpu().numpy()

# Sweep threshold
thresholds = np.linspace(0.05, 0.95, 40)
precisions, recalls, f1s = [], [], []

for thr in thresholds:
    preds_thr = (probs >= thr).astype(int)
    precisions.append(precision_score(y_bin_te, preds_thr, zero_division=0))
    recalls.append(   recall_score(   y_bin_te, preds_thr, zero_division=0))
    f1s.append(       f1_score(       y_bin_te, preds_thr, zero_division=0))

best_idx = int(np.argmax(f1s))
best_thr = thresholds[best_idx]
print(f"F1-optimal threshold: {best_thr:.2f}  (F1={f1s[best_idx]:.3f})")
print(f"Default thr=0.5:       F1={f1s[np.argmin(np.abs(thresholds-0.5))]:.3f}")

# Plot precision/recall/F1 vs threshold
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precisions, 'b-',  label='Precision')
ax.plot(thresholds, recalls,    'r-',  label='Recall')
ax.plot(thresholds, f1s,        'g-',  linewidth=2, label='F1')
ax.axvline(best_thr, color='gray', linestyle='--',
           label=f'F1-optimal thr={best_thr:.2f}')
ax.axvline(0.5, color='black', linestyle=':', label='Default thr=0.5')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/06_rw2_threshold.png', dpi=100)
plt.close()
print("Plot saved to /tmp/06_rw2_threshold.png")
print()
print("Moving threshold right: raises precision, lowers recall (more conservative).")
print("Moving threshold left:  raises recall,    lowers precision (more permissive).")

## Real-World Example 3: Multi-Label Classification

In multi-label classification each example can belong to multiple labels simultaneously
(e.g., a document is both `technology` AND `positive`).

Architecture change:
- Final layer: `sigmoid` (independent probability per label)
- Loss: `BCEWithLogitsLoss` (binary CE per label, summed)
- Evaluation: per-label F1 (precision/recall at label level)

## Comparison: TF-IDF+LR vs CNN vs LSTM

| Model | Test Accuracy | Parameters | Notes |
|---|---|---|---|
| TF-IDF + Logistic Reg | Computed below | Few (vocab × 1) | Fast, interpretable, no sequences |
| CNN (1D conv) | Computed below | ~50 K | Captures local n-gram features |
| LSTM | Computed below | ~60 K | Sequential context, slower training |

In [ ]:
# Real-World Example 3: Multi-label classification + comparison

# ── Multi-label dataset: topic (0/1/2) + sentiment (positive/negative) ────────
def make_multilabel_dataset(n: int = 200, seq_len: int = 12) -> tuple:
    """
    Each example has TWO labels:
      label[0] = topic (0=tech, 1=sports, 2=food)  — one-hot across 3 classes
      label[1] = sentiment (0=neutral, 1=positive)
    Returns encoded token sequences and binary label matrix [N, 4].
    """
    rng3 = np.random.default_rng(99)
    POS_SENT = ['great', 'love', 'amazing', 'excellent', 'happy']
    NEG_SENT = ['bad',   'hate', 'boring',  'awful',     'sad']

    X_list, y_list = [], []
    for _ in range(n):
        topic = int(rng3.integers(0, 3))
        is_pos = int(rng3.integers(0, 2))
        words = []
        for __ in range(seq_len):
            r = rng3.random()
            if r < 0.40:
                words.append(rng3.choice(TOPIC_WORDS[topic]))
            elif r < 0.60:
                pool = POS_SENT if is_pos else NEG_SENT
                words.append(rng3.choice(pool))
            else:
                words.append(rng3.choice(FILLER_WORDS))
        ids = encode_sentence(words, topic_vocab, seq_len)
        # 4-dimensional label: 3 topic one-hot + 1 sentiment
        lbl = [int(topic == 0), int(topic == 1), int(topic == 2), is_pos]
        X_list.append(ids)
        y_list.append(lbl)
    return np.array(X_list), np.array(y_list, dtype=np.float32)

X_ml, y_ml = make_multilabel_dataset(200)
sp = int(0.75 * 200)
Xml_tr, Xml_te = X_ml[:sp], X_ml[sp:]
yml_tr, yml_te = y_ml[:sp], y_ml[sp:]

Xml_tr_t = torch.tensor(Xml_tr, dtype=torch.long)
Xml_te_t = torch.tensor(Xml_te, dtype=torch.long)
yml_tr_t = torch.tensor(yml_tr, dtype=torch.float32)
yml_te_t = torch.tensor(yml_te, dtype=torch.float32)


class MultiLabelLSTM(nn.Module):
    """LSTM for multi-label classification using sigmoid outputs."""
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, n_labels: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc    = nn.Linear(hidden_dim, n_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed(x)
        _, (h, _) = self.lstm(emb)
        return self.fc(h.squeeze(0))   # [B, n_labels]  — sigmoid applied in loss

ml_model = MultiLabelLSTM(VOCAB_SZ, 32, 64, 4).to(device)
opt_ml   = optim.Adam(ml_model.parameters(), lr=1e-3)
bce_loss = nn.BCEWithLogitsLoss()

for epoch in range(100):
    ml_model.train()
    logits_ml = ml_model(Xml_tr_t.to(device))
    loss_ml   = bce_loss(logits_ml, yml_tr_t.to(device))
    opt_ml.zero_grad(); loss_ml.backward(); opt_ml.step()

ml_model.eval()
with torch.no_grad():
    logits_te_ml = ml_model(Xml_te_t.to(device))
    probs_ml = torch.sigmoid(logits_te_ml).cpu().numpy()

preds_ml = (probs_ml >= 0.5).astype(int)
label_names = ['tech', 'sports', 'food', 'positive']
print("Multi-label per-label F1:")
for i, name in enumerate(label_names):
    lf1 = f1_score(yml_te[:, i].astype(int), preds_ml[:, i], zero_division=0)
    print(f"  {name:10s}: F1 = {lf1:.3f}")

# ── Comparison: TF-IDF+LR vs CNN vs LSTM on 3-class problem ────────────────
tfidf_acc = acc           # computed in Level 1 cell
lstm_acc  = test_acc      # computed in Level 2 cell
cnn_acc   = cnn_test_acc  # computed in Level 2 cell

n_params_lstm = sum(p.numel() for p in lstm_model.parameters())
n_params_cnn  = sum(p.numel() for p in cnn_model.parameters())

print(f"\nModel comparison (3-class topic classification):")
print(f"  TF-IDF + Logistic Regression : accuracy = {tfidf_acc:.3f}  (no trainable params)")
print(f"  CNN (1D conv)                : accuracy = {cnn_acc:.3f}   ({n_params_cnn:,} params)")
print(f"  LSTM                         : accuracy = {lstm_acc:.3f}   ({n_params_lstm:,} params)")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
models = ['TF-IDF+LR', 'CNN', 'LSTM']
accs   = [tfidf_acc, cnn_acc, lstm_acc]
ax.bar(models, accs, color=['steelblue', 'darkorange', 'seagreen'])
ax.set_ylabel('Test Accuracy (3-class)')
ax.set_title('Text Classifier Comparison')
ax.set_ylim(0, 1)
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/06_comparison.png', dpi=100)
plt.close()
print("Comparison plot saved to /tmp/06_comparison.png")